# تحويل ملف إلى فيديو واسترجاعه — Google Colab
كل المعالجة تتم على خادم Colab. لا تحتاج تشغيل خادم ويب أو تثبيت برامج على جهازك. اختر **Connect** ثم شغّل خلية الإعداد أولًا، وبعدها خلية التحويل المطلوبة فقط. CPU يكفي ولا يلزم GPU.

الحد الحالي للملف الأصلي **20 MiB**. السعة نحو **2.4 MB لكل دقيقة** قبل ضغط الملف. إخراج 4K لا يزيد السعة. الاسترجاع بعد ضغط يوتيوب غير مضمون ولم يُختبر؛ الرفع التلقائي ليوتيوب غير متاح. احتفظ بالملف الأصلي.

الملفات داخل جلسة Colab مؤقتة؛ نزّل النتائج قبل انتهاء الجلسة. رفع ملف من متصفحك لا يعني أن المعالجة تعمل على جهازك.


In [ ]:
#@title 1 — تجهيز البيئة (شغّل هذه الخلية أولًا)
import sys, subprocess, shutil, tempfile, os
from pathlib import Path

REPO = "https://github.com/chatomar169-creator/video.git"
REVISION = "4a83ffd1e6c12184494e6a8d5974bd40963e3fc0"  # إصدار محرك مختبر
PROJECT = Path(tempfile.mkdtemp(prefix="binary-video-code-"))
subprocess.run(["git", "clone", "--quiet", REPO, str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--quiet", REVISION], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
# تشغيل الاختبارات في عملية جديدة تستخدم المتطلبات المثبتة.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT, check=True)
from google.colab import files

def process_file(action, source, destination, upscale=False):
    script = ("import sys,json; from codec import encode,decode; "
              "result=encode(sys.argv[2],sys.argv[3],sys.argv[4]=='true') if sys.argv[1]=='encode' else str(decode(sys.argv[2],sys.argv[3])); "
              "print(json.dumps(result))")
    result = subprocess.run([sys.executable, "-c", script, action, str(source), str(destination), str(upscale).lower()], cwd=PROJECT, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr)
    import json
    return json.loads(result.stdout)

print("البيئة جاهزة. شغّل خلية التحويل أو الاسترجاع أدناه.")


## 2 — ملف إلى فيديو
ارفع ملفًا واحدًا. اختر 4K إذا رغبت، ثم شغّل الخلية وانتظر حتى يبدأ تنزيل الفيديو.

In [ ]:
#@title تحويل ملف إلى فيديو
export_4k = False #@param {type:"boolean"}
job = Path(tempfile.mkdtemp(prefix="binary-video-encode-"))
# uploads are isolated from source code and previous jobs
previous = Path.cwd()
try:
    os.chdir(job)
    uploaded = files.upload()
finally:
    os.chdir(previous)
if len(uploaded) != 1:
    raise ValueError("اختر ملفًا واحدًا فقط ثم أعد تشغيل الخلية.")
name, data = next(iter(uploaded.items()))
if len(data) > 20 * 1024 * 1024:
    raise ValueError("الحد الأقصى 20 MiB.")
source = job / Path(name).name
source.write_bytes(data)
del uploaded, data
output_dir = job / "result"
output_dir.mkdir(exist_ok=True)
video = output_dir / "encoded.mp4"
stats = process_file("encode", source, video, export_4k)
print(f"المدة: {stats['seconds']:.2f} ثانية — حجم الفيديو: {stats['video_bytes'] / 1048576:.2f} MiB")
print("النتيجة:", video)
files.download(str(video))


## 3 — فيديو إلى الملف الأصلي
هذه خلية مستقلة بعد الإعداد. ارفع فيديو أنتجته الأداة. لن تُنزّل النتيجة إلا بعد نجاح التحقق من بصمة الملف.

In [ ]:
#@title استرجاع الملف من الفيديو
job = Path(tempfile.mkdtemp(prefix="binary-video-decode-"))
previous = Path.cwd()
try:
    os.chdir(job)
    uploaded = files.upload()
finally:
    os.chdir(previous)
if len(uploaded) != 1:
    raise ValueError("اختر فيديو واحدًا فقط ثم أعد تشغيل الخلية.")
name, data = next(iter(uploaded.items()))
if len(data) > 256 * 1024 * 1024:
    raise ValueError("حد رفع الفيديو في هذه النسخة 256 MiB.")
source = job / Path(name).name
source.write_bytes(data)
del uploaded, data
output_dir = job / "recovered"
output_dir.mkdir(exist_ok=True)
recovered = process_file("decode", source, output_dir)
print("تم التحقق من SHA-256: الملف مطابق للأصل.")
print("النتيجة:", recovered)
files.download(recovered)


## ملاحظات
- لا تستخدم Run all لأنه سيطلب رفع ملف ثم فيديو؛ اختر العملية المطلوبة.
- عند انقطاع الجلسة شغّل الإعداد مرة أخرى وارفع الملف مجددًا.
- إذا ظهر خطأ Incomplete data، فالفيديو فقد بيانات أو ليس بالتنسيق المطلوب؛ جرّب الفيديو الأصلي.
- هذا Notebook يستخدم إصدارًا ثابتًا من المحرك لثبات النتائج. [المصدر والتفاصيل](https://github.com/chatomar169-creator/video).
